# 6. Proposition Chunking

Rewrite passages into atomic, self-contained facts (propositions) and index those.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

## Why this matters

A normal chunk mixes several facts and has pronouns like *it* or *this model* whose meaning is only clear from earlier text. **Propositional chunking** (Dense X Retrieval) rewrites text into atomic, self-contained facts. Each proposition is one idea, fully de-referenced, so its embedding is sharp and it can match a specific question precisely.

## How it works

1. Take each chunk
2. Ask the LLM to decompose it into atomic statements and resolve pronouns
3. Embed each proposition separately
4. Search propositions; map hits back to their parent chunk for context
5. Generate from the parent chunks (or the propositions themselves)

```text
 chunk: "It produces 384-dimensional vectors and is popular because it is small."
   -> LLM ->  "all-MiniLM-L6-v2 produces 384-dimensional vectors."
              "all-MiniLM-L6-v2 is popular for local experiments because it is small and fast."
   -> embed each -> search -> parent chunk -> LLM
```

## Prerequisites

Needs a running Ollama server (`ollama serve`) with `llama3.2` pulled, plus the shared helpers in `../common.py`.

## Setup

Load the handbook, split it into ~500-character chunks (50 overlap) and index them. Every notebook starts from this same baseline so results are comparable.

In [1]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

W1028 10:25:20.109000 6754 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


19 chunks indexed


A **proposition** is a single fact that makes sense on its own - pronouns resolved, no hidden references.

In [2]:
def propositions(chunk):
    data = llm_json(
        "Decompose the passage into short, self-contained factual statements. Replace pronouns with the "
        "entity they refer to. Reply with a JSON list of strings.\n\nPassage: " + chunk
    )
    return [p for p in (data or []) if isinstance(p, str)]

print(propositions(chunks[0]))

['The Applied AI Handbook', 'Retrieval-Augmented Generation (RAG) combines a retriever with a language model.', 'A retriever is used instead of relying only on what the model memorised during training.', 'The system first searches an external knowledge base and then passes the retrieved passages to the model as context.', 'This keeps answers grounded in up-to-date, private or domain-specific data without retraining the model.', 'A basic RAG pipeline has four stages: ingest documents, split them into', 'Ingesting documents involves processing and storing the data for later use.']


In [3]:
props, src = [], []
for i, chunk in enumerate(chunks):
    for p in propositions(chunk):
        props.append(p)
        src.append({"chunk": i})
prop_store = build_store(props, src)
print(len(props), "propositions")

107 propositions


In [4]:
query = "What is the constant in the Reciprocal Rank Fusion formula?"
print("--- propositions"); show(prop_store.search(query, 3))
print("--- plain chunks"); show(store.search(query, 3))

--- propositions


0.666  Reciprocal Rank Fusion adds 1 divided by (60 plus rank) for every list in which a document appears....
0.414  Hybrid search merges rankings with a weighted score or Reciprocal Rank Fusion....
0.325  The average of 1 divided by the rank of the first correct result is used for retrieval measurement....
--- plain chunks
0.227  r and outputs a relevance score, which is far more accurate than comparing two independent vectors. Because cross-encoders are slow, they are applied only to th...
0.221  t paraphrases and meaning. Hybrid search runs both and merges the rankings, either with a weighted score or with Reciprocal Rank Fusion, which adds 1 divided by...
0.153  is usually measured with cosine similarity, which compares direction rather than length. If the vectors are normalised to unit length, cosine similarity equals ...


## Quality control
LLM decomposition can drop facts or add new ones. Spot-check by comparing a chunk with its propositions.

In [5]:
i = 1
print("CHUNK:\n", chunks[i], "\n")
for p in propositions(chunks[i]): print(" -", p)

CHUNK:
 has four stages: ingest documents, split them into chunks, index the chunks as embeddings, and at query time retrieve the closest chunks and generate an answer from them.

## Embeddings

An embedding model maps text to a dense vector so that texts with similar meaning land close together. The all-MiniLM-L6-v2 model produces 384-dimensional vectors and is popular for local experiments because it is small and fast. Similarity between two vectors is usually measured with cosine similarity, which 

 - The passage has four stages: ingest documents, split them into chunks, index the chunks as embeddings, and at query time retrieve the closest chunks and generate an answer from them.
 - An embedding model maps text to a dense vector so that texts with similar meaning land close together.
 - The all-MiniLM-L6-v2 model produces 384-dimensional vectors and is popular for local experiments because it is small and fast.
 - Similarity between two vectors is usually measured with cosine simi

## Do propositions retrieve better than chunks?
Propositions are paraphrased, so substring matching on them would be unfair. We map each proposition hit back to its parent chunk and check the evidence there.

In [6]:
def prop_search(query, k=3):
    seen, out = set(), []
    for h in prop_store.search(query, k * 4):
        c = h["meta"]["chunk"]
        if c not in seen:
            seen.add(c); out.append({"idx": c, "text": chunks[c], "score": h["score"], "meta": {}})
    return out[:k]

compare_retrievers({"fixed chunks (500)": store.search, "propositions -> parent chunk": prop_search}, k=3)

,hit@1,hit@3,MRR,ranks
method,,,,
fixed chunks (500),0.8,1.0,0.89,"[1, 2, 2, 1, 3, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, ..."
propositions -> parent chunk,0.9,1.0,0.95,"[1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."


## Inspect the index
How many propositions per chunk, and how long are they?

In [7]:
import pandas as pd
pd.DataFrame({"chars": [len(p) for p in props]}).describe().round(1).T

,count,mean,std,min,25%,50%,75%,max
chars,107.0,77.2,31.3,16.0,55.0,75.0,96.0,182.0


## Trade-offs

| Strengths | Weaknesses |
|---|---|
| Sharper embeddings; precise matches for specific facts | Expensive: one LLM call per chunk at indexing time |
| Pronouns and references are resolved | Risk of LLM dropping or inventing facts |
| Great for question answering over dense factual text | Index size grows 3-10x |

## When to use it

Fact-dense corpora (encyclopaedic, policy, technical docs) where users ask narrow factual questions and indexing cost is acceptable.

## Experiments to try

- Add a second LLM pass that grades each proposition for accuracy/completeness.
- Retrieve propositions but send the parent chunk to the generator.
- Compare index size and hit rate against fixed chunks.

## Key takeaways

- Atomic, de-referenced facts embed better.
- Always keep a pointer to the source chunk.
- Validate LLM-generated propositions.

---
**Next:** [7. Evaluation with DeepEval](../07_evaluation_deep_eval/evaluation_deep_eval.ipynb)